# Master Data Governance & Operations Analytics System.

Step 1: Environment Setup & Operational Data Generation
Explanation of Step 1
Before enforcing data governance or building Power BI dashboards, we need an operational business dataset with master data tables and intentional data quality defects.

In this step, we will write a Python script in Google Colab to:

Define Master Data Schema: Create relational structures for Customers (Dim_Customer), Products (Dim_Product), and Operational Transactions (Fact_Transactions).

Inject Controlled Governance Defects: Simulate real-world operational data issues (e.g., duplicate customer keys, missing metadata fields, negative transaction amounts, invalid email formats) to test our data governance and validation engine in Step 2.

Save Raw Operational Data: Write the uncleaned raw tables into SQLite and CSV formats within Google Colab

In [1]:
import sqlite3
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)

# --- 1. GENERATE DIM_CUSTOMER (Master Customer Data) ---
n_customers = 100
customer_ids = [f"CUST-{1000 + i}" for i in range(n_customers)]
regions = ["APAC", "EMEA", "NA", "LATAM"]

customer_data = {
    "CustomerID": customer_ids,
    "CustomerName": [f"Client_{i}" for i in range(n_customers)],
    "Region": np.random.choice(regions, n_customers),
    "ContactEmail": [f"contact_client_{i}@company.com" for i in range(n_customers)],
    "AccountStatus": np.random.choice(
        ["Active", "Pending", "Inactive"], n_customers, p=[0.8, 0.1, 0.1]
    ),
}
df_customers = pd.DataFrame(customer_data)

# Inject Data Governance Defects into Master Customer Data
df_customers.loc[5, "ContactEmail"] = "invalid_email_at_company.com"  # Missing @
df_customers.loc[12, "CustomerID"] = "CUST-1000"  # Duplicate Key defect


# --- 2. GENERATE DIM_PRODUCT (Master Product/Service Data) ---
n_products = 20
product_ids = [f"PROD-{200 + i}" for i in range(n_products)]
categories = [
    "Advisory",
    "Assurance",
    "Tax Services",
    "Business Process Automation",
]

product_data = {
    "ProductID": product_ids,
    "ProductName": [f"Service_Package_{i}" for i in range(n_products)],
    "Category": np.random.choice(categories, n_products),
    "StandardSLA_Days": np.random.choice([3, 5, 7, 10], n_products),
}
df_products = pd.DataFrame(product_data)


# --- 3. GENERATE FACT_TRANSACTIONS (Operational Transactions) ---
n_transactions = 500
dates = pd.date_range(start="2025-01-01", end="2026-06-30", periods=n_transactions)

transaction_data = {
    "TransactionID": [f"TXN-{10000 + i}" for i in range(n_transactions)],
    "TransactionDate": dates,
    "CustomerID": np.random.choice(
        customer_ids + ["CUST-9999"], n_transactions
    ),  # 'CUST-9999' is an Orphan Key defect
    "ProductID": np.random.choice(product_ids, n_transactions),
    "RevenueUSD": np.round(np.random.uniform(5000, 150000, n_transactions), 2),
    "ActualSLA_Days": np.random.randint(1, 14, n_transactions),
    "OperationalStatus": np.random.choice(
        ["Completed", "Delayed", "In Review"], n_transactions, p=[0.7, 0.2, 0.1]
    ),
}
df_transactions = pd.DataFrame(transaction_data)

# Inject Data Quality Defects into Transactions
df_transactions.loc[15, "RevenueUSD"] = -15000.00  # Negative Revenue defect
df_transactions.loc[45, "RevenueUSD"] = np.nan  # Missing Revenue value defect


# --- 4. LOAD RAW DATA INTO SQLITE DATABASE ---
conn = sqlite3.connect("operations_raw.db")

df_customers.to_sql("raw_dim_customer", conn, if_exists="replace", index=False)
df_products.to_sql("raw_dim_product", conn, if_exists="replace", index=False)
df_transactions.to_sql(
    "raw_fact_transactions", conn, if_exists="replace", index=False
)

print("Step 1 Complete: Raw operational database 'operations_raw.db' successfully created.")
print(f"Raw Customers Record Count: {len(df_customers)}")
print(f"Raw Products Record Count: {len(df_products)}")
print(f"Raw Transactions Record Count: {len(df_transactions)}")

conn.close()

Step 1 Complete: Raw operational database 'operations_raw.db' successfully created.
Raw Customers Record Count: 100
Raw Products Record Count: 20
Raw Transactions Record Count: 500


Step 2: Data Governance Rules & Audit Logging Engine
Explanation of Step 2
Now that we have raw operational tables with intentional anomalies, we need to build an automated Data Governance & Data Quality Engine.

In this step, we will write a Python script in Google Colab that:

Applies Governance Validation Rules:

Master Data Rule 1: Detect duplicate primary keys (CustomerID must be unique).

Master Data Rule 2: Validate contact formats (ContactEmail must contain @ and .).

Transaction Rule 1: Detect financial anomalies (RevenueUSD cannot be negative or null).

Transaction Rule 2: Enforce Referential Integrity (flag orphan keys where CustomerID does not exist in Dim_Customer).

Generates Audit Logs: Automatically routes any record that fails these rules into an etl_audit_log table along with a reason code and timestamp.

Outputs Clean Production Tables: Loads clean records into production-ready SQLite tables (dim_customer, dim_product, fact_transactions) and exports them as a clean Excel database for Power BI ingestion.

In [2]:
from datetime import datetime
import sqlite3
import numpy as np
import pandas as pd

# Connect to the raw SQLite database created in Step 1
conn = sqlite3.connect("operations_raw.db")

df_raw_cust = pd.read_sql_query("SELECT * FROM raw_dim_customer", conn)
df_raw_prod = pd.read_sql_query("SELECT * FROM raw_dim_product", conn)
df_raw_txn = pd.read_sql_query("SELECT * FROM raw_fact_transactions", conn)

# Data structure to hold audit records
audit_logs = []


def log_audit(record_id, table_name, field_name, defect_type, severity):
    audit_logs.append(
        {
            "LogTimestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "RecordID": record_id,
            "TableName": table_name,
            "FieldName": field_name,
            "DefectType": defect_type,
            "Severity": severity,
        }
    )


# --- 1. VALIDATE DIM_CUSTOMER ---
clean_customers = []
seen_cust_ids = set()

for idx, row in df_raw_cust.iterrows():
    cust_id = row["CustomerID"]
    email = row["ContactEmail"]
    has_defect = False

    # Check 1: Duplicate Key
    if cust_id in seen_cust_ids:
        log_audit(
            cust_id,
            "Dim_Customer",
            "CustomerID",
            "Duplicate Primary Key",
            "High",
        )
        has_defect = True
    else:
        seen_cust_ids.add(cust_id)

    # Check 2: Email Format
    if "@" not in str(email) or "." not in str(email):
        log_audit(
            cust_id,
            "Dim_Customer",
            "ContactEmail",
            "Invalid Format Pattern",
            "Medium",
        )
        has_defect = True

    if not has_defect:
        clean_customers.append(row)

df_clean_cust = pd.DataFrame(clean_customers)


# --- 2. VALIDATE DIM_PRODUCT ---
# Products pass through directly as baseline clean master data
df_clean_prod = df_raw_prod.copy()


# --- 3. VALIDATE FACT_TRANSACTIONS ---
clean_transactions = []
valid_customer_ids = set(df_clean_cust["CustomerID"])

for idx, row in df_raw_txn.iterrows():
    txn_id = row["TransactionID"]
    cust_id = row["CustomerID"]
    revenue = row["RevenueUSD"]
    has_defect = False

    # Check 1: Referential Integrity (Orphan Key)
    if cust_id not in valid_customer_ids:
        log_audit(
            txn_id,
            "Fact_Transactions",
            "CustomerID",
            "Orphan Foreign Key",
            "Critical",
        )
        has_defect = True

    # Check 2: Financial Integrity (Negative Revenue)
    if pd.isna(revenue) or revenue < 0:
        log_audit(
            txn_id,
            "Fact_Transactions",
            "RevenueUSD",
            "Invalid/Negative Financial Value",
            "High",
        )
        has_defect = True

    if not has_defect:
        clean_transactions.append(row)

df_clean_txn = pd.DataFrame(clean_transactions)
df_audit_log = pd.DataFrame(audit_logs)


# --- 4. PERSIST CLEAN DATA & AUDIT LOGS ---
df_clean_cust.to_sql("dim_customer", conn, if_exists="replace", index=False)
df_clean_prod.to_sql("dim_product", conn, if_exists="replace", index=False)
df_clean_txn.to_sql("fact_transactions", conn, if_exists="replace", index=False)
df_audit_log.to_sql("etl_audit_log", conn, if_exists="replace", index=False)

# Export clean database for Power BI Ingestion
with pd.ExcelWriter("Operations_Governance_Dataset.xlsx") as writer:
    df_clean_cust.to_excel(writer, sheet_name="Dim_Customer", index=False)
    df_clean_prod.to_excel(writer, sheet_name="Dim_Product", index=False)
    df_clean_txn.to_excel(writer, sheet_name="Fact_Transactions", index=False)
    df_audit_log.to_excel(writer, sheet_name="ETL_Audit_Log", index=False)

print("Step 2 Complete: Governance Engine Executed.")
print(f"Total Defects Isolated & Logged: {len(df_audit_log)}")
print("\n--- DATA GOVERNANCE AUDIT LOG SUMMARY ---")
print(
    df_audit_log[
        ["RecordID", "TableName", "FieldName", "DefectType", "Severity"]
    ]
)

conn.close()

Step 2 Complete: Governance Engine Executed.
Total Defects Isolated & Logged: 21

--- DATA GOVERNANCE AUDIT LOG SUMMARY ---
     RecordID          TableName     FieldName  \
0   CUST-1005       Dim_Customer  ContactEmail   
1   CUST-1000       Dim_Customer    CustomerID   
2   TXN-10015  Fact_Transactions    RevenueUSD   
3   TXN-10045  Fact_Transactions    RevenueUSD   
4   TXN-10111  Fact_Transactions    CustomerID   
5   TXN-10157  Fact_Transactions    CustomerID   
6   TXN-10194  Fact_Transactions    CustomerID   
7   TXN-10201  Fact_Transactions    CustomerID   
8   TXN-10210  Fact_Transactions    CustomerID   
9   TXN-10220  Fact_Transactions    CustomerID   
10  TXN-10246  Fact_Transactions    CustomerID   
11  TXN-10267  Fact_Transactions    CustomerID   
12  TXN-10299  Fact_Transactions    CustomerID   
13  TXN-10309  Fact_Transactions    CustomerID   
14  TXN-10334  Fact_Transactions    CustomerID   
15  TXN-10351  Fact_Transactions    CustomerID   
16  TXN-10361  Fact_Transa

Step 3: SQL Root Cause Analysis (RCA) & Power BI DAX Data Model
Explanation of Step 3
Now that clean data is written to Operations_Governance_Dataset.xlsx and the SQLite database, we need to:

Execute SQL Root Cause Analysis (RCA): Query the etl_audit_log table in SQLite using Advanced SQL (CTEs and aggregations) to quantify defect rates by table and severity.

Define the Star Schema Data Model: Structure relational joins between our dimension and fact tables for Power BI.

Write Advanced DAX Measures: Create business intelligence metrics for operational SLA compliance, total portfolio revenue, and governance error rates.

Part 1: Execute SQL RCA in Google Colab
Run this code in a new cell in Google Colab to generate your Root Cause Analysis metrics directly from the audit log:

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("operations_raw.db")

# Advanced SQL CTE query for Governance Defect Distribution & Impact
sql_rca_query = """
WITH DefectSummary AS (
    SELECT
        TableName,
        DefectType,
        Severity,
        COUNT(RecordID) AS TotalDefects
    FROM etl_audit_log
    GROUP BY TableName, DefectType, Severity
),
TotalLogCount AS (
    SELECT COUNT(*) AS TotalSystemDefects FROM etl_audit_log
)
SELECT
    ds.TableName,
    ds.DefectType,
    ds.Severity,
    ds.TotalDefects,
    ROUND((CAST(ds.TotalDefects AS FLOAT) / tlc.TotalSystemDefects) * 100, 2) AS DefectPercentage
FROM DefectSummary ds
CROSS JOIN TotalLogCount tlc
ORDER BY ds.TotalDefects DESC;
"""

df_rca = pd.read_sql_query(sql_rca_query, conn)
print("--- SQL ROOT CAUSE ANALYSIS (RCA) METRICS ---")
print(df_rca.to_string(index=False))

conn.close()

--- SQL ROOT CAUSE ANALYSIS (RCA) METRICS ---
        TableName                       DefectType Severity  TotalDefects  DefectPercentage
Fact_Transactions               Orphan Foreign Key Critical            17             80.95
Fact_Transactions Invalid/Negative Financial Value     High             2              9.52
     Dim_Customer            Duplicate Primary Key     High             1              4.76
     Dim_Customer           Invalid Format Pattern   Medium             1              4.76


Step 4: Download Dataset from Colab & Build Power BI Dashboard
1. Download the Dataset from Google Colab
Run this code in a new cell in Google Colab to download the Excel file directly to your computer:

In [5]:
from google.colab import files

files.download("Operations_Governance_Dataset.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>